# Weather History Silver Pipeline

Pipeline end-to-end cho Open-Meteo Historical Forecast:

Committed Bronze
→ Processed Ingestion Control
→ Pending Ingestion Selection
→ Bulk Load
→ Transformation
→ Raw Data Quality
→ Ingestion Validation
→ Overlap Reconciliation
→ Final Data Quality
→ Existing Silver Conflict Check
→ Delta MERGE
→ Mark Ingestion Processed
→ Persistence Verification

Nếu không còn pending ingestion, pipeline kết thúc thành công với trạng thái `NO_OP`.

In [0]:
import os

account_name = os.getenv("AZURE_STORAGE_ACCOUNT_NAME")
tenant_id = os.getenv("AZURE_TENANT_ID")
client_id = os.getenv("AZURE_CLIENT_ID")
client_secret = os.getenv("AZURE_CLIENT_SECRET")

endpoint = f"{account_name}.dfs.core.windows.net"

spark.conf.set(
    f"fs.azure.account.auth.type.{endpoint}",
    "OAuth"
)

spark.conf.set(
    f"fs.azure.account.oauth.provider.type.{endpoint}",
    "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider"
)

spark.conf.set(
    f"fs.azure.account.oauth2.client.id.{endpoint}",
    client_id
)

spark.conf.set(
    f"fs.azure.account.oauth2.client.secret.{endpoint}",
    client_secret
)

spark.conf.set(
    f"fs.azure.account.oauth2.client.endpoint.{endpoint}",
    f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
)

In [0]:
import importlib
import fastorder.transformation.silver.weather.history_hourly as history_hourly

importlib.reload(history_hourly)

In [0]:
from fastorder.storage.adls_client import get_adls_service_client


from fastorder.transformation.silver.weather.history_hourly import (
    discover_committed_history_ingestions,
    get_processed_history_ingestion_ids,
    find_pending_history_ingestions,
    load_pending_history_bronze,
    transform_history_hourly,
    profile_history_data_quality,
    assert_history_data_quality,
    profile_history_validation,
    assert_history_validation,
    resolve_history_overlaps,
    prepare_history_silver_output,
    assert_no_conflicting_history_with_silver,
    merge_history_silver,
    build_history_processed_ingestions,
    mark_history_ingestions_processed,
    extract_history_ingestion_id_from_path,
)

## 1. Pipeline Configuration

In [0]:
BRONZE_HISTORY_ROOT = (
    "weather/open_meteo/historical_forecast"
)

BRONZE_ABFSS_ROOT = (
    "abfss://bronze@fastorderdatalake.dfs.core.windows.net"
)

SILVER_HISTORY_PATH = (
    "abfss://silver@fastorderdatalake.dfs.core.windows.net/"
    "weather/history_hourly"
)

SILVER_HISTORY_CONTROL_PATH = (
    "abfss://silver@fastorderdatalake.dfs.core.windows.net/"
    "control/weather_history_processed_ingestions"
)

## 2. Initialize Storage Client

In [0]:
service_client = get_adls_service_client()

bronze_client = (
    service_client
    .get_file_system_client("bronze")
)

## 3. End-to-End Pipeline Orchestration

Pipeline xử lý toàn bộ pending Historical ingestion trong một batch.

Hai trạng thái thành công:

- `SUCCESS`: có ingestion mới được xử lý và persist.
- `NO_OP`: không còn ingestion mới cần xử lý.

In [0]:
def run_weather_history_silver_pipeline():

    committed_ingestion_paths = (
        discover_committed_history_ingestions(
            bronze_client=bronze_client,
            history_root=BRONZE_HISTORY_ROOT,
        )
    )


    processed_ingestion_ids = (
        get_processed_history_ingestion_ids(
            spark=spark,
            control_path=SILVER_HISTORY_CONTROL_PATH,
        )
    )



    pending_ingestion_paths = (
        find_pending_history_ingestions(
            committed_ingestion_paths=
                committed_ingestion_paths,
            processed_ingestion_ids=
                processed_ingestion_ids,
        )
    )

    print(
        f"Committed Bronze : "
        f"{len(committed_ingestion_paths)}"
    )

    print(
        f"Processed Control: "
        f"{len(processed_ingestion_ids)}"
    )

    print(
        f"Pending          : "
        f"{len(pending_ingestion_paths)}"
    )


    if not pending_ingestion_paths:

        return {
            "status": "NO_OP",
            "committed_ingestion_count":
                len(committed_ingestion_paths),
            "processed_ingestion_count":
                len(processed_ingestion_ids),
            "pending_ingestion_count": 0,
            "raw_row_count": 0,
            "resolved_row_count": 0,
        }


    df_response, df_metadata = (
        load_pending_history_bronze(
            spark=spark,
            pending_ingestion_paths=
                pending_ingestion_paths,
            bronze_abfss_root=
                BRONZE_ABFSS_ROOT,
        )
    )


    df_history_raw_candidate = (
        transform_history_hourly(
            df_response=df_response,
            df_metadata=df_metadata,
        )
    )

    raw_row_count = (
        df_history_raw_candidate.count()
    )



    raw_dq_result = (
        profile_history_data_quality(
            df_history_raw_candidate
        )
    )

    assert_history_data_quality(
        raw_dq_result,
        allow_duplicate_grain=True,
    )


    validation_result = (
        profile_history_validation(
            df=df_history_raw_candidate,
            df_metadata=df_metadata,
            pending_ingestion_paths=
                pending_ingestion_paths,
        )
    )

    assert_history_validation(
        validation_result
    )

    df_history_resolved_candidate = (
        resolve_history_overlaps(
            df_history_raw_candidate
        )
    )

    resolved_row_count = (
        df_history_resolved_candidate.count()
    )


    final_dq_result = (
        profile_history_data_quality(
            df_history_resolved_candidate
        )
    )

    assert_history_data_quality(
        final_dq_result
    )


    df_history_silver_output = (
        prepare_history_silver_output(
            df_history_resolved_candidate
        )
    )


    assert_no_conflicting_history_with_silver(
        spark=spark,
        df_incoming=
            df_history_silver_output,
        silver_path=
            SILVER_HISTORY_PATH,
    )


    merge_history_silver(
        spark=spark,
        df=df_history_silver_output,
        silver_path=SILVER_HISTORY_PATH,
    )


    df_processed_ingestions = (
        build_history_processed_ingestions(
            df_metadata
        )
    )



    mark_history_ingestions_processed(
        spark=spark,
        df_processed_ingestions=
            df_processed_ingestions,
        control_path=
            SILVER_HISTORY_CONTROL_PATH,
    )

    processed_ids_after_write = (
        get_processed_history_ingestion_ids(
            spark=spark,
            control_path=
                SILVER_HISTORY_CONTROL_PATH,
        )
    )

    current_ingestion_ids = {
        extract_history_ingestion_id_from_path(
            path
        )
        for path in pending_ingestion_paths
    }

    missing_processed_ids = (
        current_ingestion_ids
        - processed_ids_after_write
    )

    if missing_processed_ids:
        raise ValueError(
            "Historical processing-state verification "
            "FAILED: "
            f"{missing_processed_ids}"
        )


    df_persisted = (
        spark.read
        .format("delta")
        .load(SILVER_HISTORY_PATH)
    )

    duplicate_grain_count = (
        df_persisted
        .groupBy(
            "warehouse_id",
            "weather_time",
        )
        .count()
        .filter("count > 1")
        .count()
    )

    if duplicate_grain_count != 0:
        raise ValueError(
            "Historical Silver persistence verification "
            "FAILED: duplicate business grain."
        )

    return {
        "status": "SUCCESS",
        "committed_ingestion_count":
            len(committed_ingestion_paths),
        "processed_ingestion_count_before":
            len(processed_ingestion_ids),
        "pending_ingestion_count":
            len(pending_ingestion_paths),
        "raw_row_count":
            raw_row_count,
        "resolved_row_count":
            resolved_row_count,
        "overlap_rows_removed":
            raw_row_count - resolved_row_count,
        "expected_total_rows":
            validation_result["expected_total_rows"],
    }

In [0]:
pipeline_result = (
    run_weather_history_silver_pipeline()
)

pipeline_result